In [1]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

## CMAPSSData Preprocessing

In [2]:
#Eliminar sensores constantes
def eliminar_constantes(df, umbral_std=0.01):
    sensor_cols = [c for c in df.columns if c.startswith("s")]
    std = df[sensor_cols].std()
    constantes = std[std < umbral_std].index.tolist()
    print(f"Sensores eliminados por baja varianza: {constantes}")
    return df.drop(columns=constantes), constantes

#Calcular y añadir la etiqueta RUL
def add_rul_train(df, rul_max=125):
    """
    RUL real = ciclos restantes hasta el fallo.
    Se trunca a rul_max (cap lineal): permite que el modelo se centre en los ciclos 
    cercanos al fallo. No hay necesidad de predecir el fallo desde el inicio del trayecto.
    """
    max_ciclo = df.groupby("motor_id")["ciclo"].max().rename("ciclo_max")
    df = df.join(max_ciclo, on="motor_id")
    df["rul"] = (df["ciclo_max"] - df["ciclo"]).clip(upper=rul_max)
    df = df.drop(columns=["ciclo_max"])
    return df

def add_rul_test(df, rul_finales, rul_max=125):
    """
    En test, NASA proporciona cuántos ciclos quedaban al final
    del fragmento observado. Se suman los ciclos restantes y se calcula
    el RUL para cada ciclo de cada motor
    """
    rul_map = {
        motor_id + 1: rul
        for motor_id, rul in enumerate(rul_finales["rul_final"].values)
    }
    max_ciclo = df.groupby("motor_id")["ciclo"].max()
    
    def rul_para_motor(row):
        rul_final = rul_map[row["motor_id"]]
        ciclos_restantes = (max_ciclo[row["motor_id"]] - row["ciclo"]) + rul_final
        return min(ciclos_restantes, rul_max)
    
    df["rul"] = df.apply(rul_para_motor, axis=1)
    return df

def norm(train_df, test_df, feature_cols):
    """
    MinMaxScaler ajustado SOLO sobre train.
    Aplica la misma transformación a test sin re-ajustar.
    """
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
    test_df[feature_cols] = scaler.transform(test_df[feature_cols]) 
    
    return train_df, test_df, scaler

def crear_ventanas(df, feature_cols, window_size=30):
    """
    Para cada motor, recorre sus ciclos con una ventana de tamaño window_size.
    Si el motor tiene menos ciclos que window_size, se rellena con padding
    al principio (ceros)
    
    Devuelve:
        X: (N, window_size, n_features)
        y: (N,)  — RUL en el último ciclo de cada ventana
    """
    X_list, y_list = [], []
    
    for motor_id, grupo in df.groupby("motor_id"):
        datos = grupo[feature_cols].values   # (ciclos, n_features)
        etiqueta = grupo["rul"].values           # (ciclos,)
        n_ciclos = len(datos)
        
        if n_ciclos < window_size:
            # Padding al principio con ceros
            pad = np.zeros((window_size - n_ciclos, datos.shape[1]))
            datos = np.vstack([pad, datos])
            etiqueta = np.concatenate([np.full(window_size - n_ciclos, etiqueta[0]), etiqueta])
            n_ciclos = window_size
        
        for i in range(n_ciclos - window_size + 1):
            X_list.append(datos[i : i + window_size])
            y_list.append(etiqueta[i + window_size - 1])   # RUL del último ciclo
    
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

def crear_ventanas_test(df, feature_cols, window_size=30):
    """
    Se usa la última ventana de cada motor
    """
    X_list, y_list = [], []
    
    for motor_id, grupo in df.groupby("motor_id"):
        datos = grupo[feature_cols].values
        etiqueta = grupo["rul"].values
        n_ciclos = len(datos)
        
        if n_ciclos >= window_size:
            X_list.append(datos[-window_size:])
        else:
            pad = np.zeros((window_size - n_ciclos, datos.shape[1]))
            ventana = np.vstack([pad, datos])
            X_list.append(ventana)
        
        y_list.append(etiqueta[-1])
    
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


In [3]:
def preparar_dataset(ruta_base="CMAPSSData", subset="FD001", window_size=30, rul_max=125):

    COLS = (
    ["motor_id", "ciclo"] +
    [f"op_{i}" for i in range(1, 4)] +          # 3 configuraciones operacionales
    [f"s{i}" for i in range(1, 22)]             # 21 sensores
    )
    
    train_data = pd.read_csv('../CMAPSSData/train_FD001.txt', sep=r"\s+", header=None, names=COLS)
    test_data = pd.read_csv('../CMAPSSData/test_FD001.txt',  sep=r"\s+", header=None, names=COLS)
    rul = pd.read_csv('../CMAPSSData/RUL_FD001.txt',   sep=r"\s+", header=None, names=["rul_final"])
    
    # Añadir etiquetas RUL
    train_df = add_rul_train(train_data.copy(), rul_max=rul_max)
    test_df  = add_rul_test(test_data.copy(), rul, rul_max=rul_max)

    # Eliminar sensores constantes (calculado sobre train)
    sensor_cols = [c for c in train_df.columns if c.startswith("s")]
    std = train_df[sensor_cols].std()
    constantes = std[std < 0.01].index.tolist()
    train_df = train_df.drop(columns=constantes)
    test_df  = test_df.drop(columns=constantes)

    # Columnas que entran al modelo (sensores + config. operacional)
    feature_cols = [c for c in train_df.columns
                    if c.startswith("s") or c.startswith("op")]

    # Normalizar
    train_df, test_df, scaler = norm(train_df, test_df, feature_cols)

    # Ventanas
    X_train_full, y_train_full = crear_ventanas(train_df, feature_cols, window_size)
    X_test,  y_test  = crear_ventanas_test(test_df, feature_cols, window_size)

    # Split validación desde train
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.15, random_state=42
    )

    print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
    print(f"X_val:   {X_val.shape}    y_val:   {y_val.shape}")
    print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")
    print(f"Features usadas: {len(feature_cols)} — {feature_cols}")

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler, feature_cols


In [4]:
# Ejemplo de uso:
(X_tr, y_tr), (X_val, y_val), (X_te, y_te), scaler, feats = preparar_dataset()
# model.fit(X_tr, y_tr, validation_data=(X_val, y_val), ...)

X_train: (15071, 30, 17)  y_train: (15071,)
X_val:   (2660, 30, 17)    y_val:   (2660,)
X_test:  (100, 30, 17)   y_test:  (100,)
Features usadas: 17 — ['op_1', 'op_2', 'op_3', 's2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


## 1D_CNN

In [5]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

In [6]:
#Construcción CNN reutilizable
def bloque_conv(x, filtros, kernel_size, dilation_rate=1, nombre=""):
    """
    Unidad básica: Conv1D → BatchNorm → ReLU → Dropout ligero.
    dilation_rate > 1 amplía el campo receptivo sin aumentar parámetros.
    """
    x = layers.Conv1D(
        filters      = filtros,
        kernel_size  = kernel_size,
        padding      = "same",          # conserva la longitud temporal
        dilation_rate= dilation_rate,
        use_bias     = False,           # BatchNorm tiene su propio bias
        name         = f"conv_{nombre}"
    )(x)
    x = layers.BatchNormalization(name=f"bn_{nombre}")(x)
    x = layers.Activation("relu", name=f"relu_{nombre}")(x)
    return x


def build_cnn_block(inp, filtros=64, nombre_base="cnn"):
    """
    Tres capas convolucionales con kernel sizes distintos (3, 5, 3)
    y la tercera con dilation=2 para capturar tendencias más largas.

    Entrada:  (batch, window_size, n_sensores)
    Salida:   (batch, window_size, filtros)  ← misma longitud temporal
    """
    # Capa 1 — patrones locales finos (3 ciclos)
    x = bloque_conv(inp, filtros, kernel_size=3, nombre=f"{nombre_base}_1")

    # Capa 2 — patrones medios (5 ciclos)
    x = bloque_conv(x,   filtros, kernel_size=5, nombre=f"{nombre_base}_2")

    # Capa 3 — patrones dilatados: campo receptivo = 5 ciclos con solo 3 pesos
    x = bloque_conv(x,   filtros, kernel_size=3,
                    dilation_rate=2, nombre=f"{nombre_base}_3")

    # Conexión residual: si el número de canales cambia, proyectar con Conv 1×1
    if inp.shape[-1] != filtros:
        shortcut = layers.Conv1D(filtros, 1, padding="same",
                                 use_bias=False, name=f"{nombre_base}_proj")(inp)
        shortcut = layers.BatchNormalization(name=f"{nombre_base}_bn_proj")(shortcut)
    else:
        shortcut = inp

    x = layers.Add(name=f"{nombre_base}_residual")([x, shortcut])
    x = layers.Activation("relu", name=f"{nombre_base}_out")(x)
    return x

In [7]:
def build_cnn_block(inp, filtros=64):
    """
    Versión sin residual — más fácil de leer, funciona bien para CMAPSS.
    La dimensión temporal se conserva gracias a padding='same'.
    NO hay Flatten porque la salida va al BiLSTM.
    """
    x = layers.Conv1D(filtros, kernel_size=3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)

    x = layers.Conv1D(filtros, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)

    x = layers.Conv1D(filtros, kernel_size=3, padding="same",
                      dilation_rate=2, activation="relu")(x)
    x = layers.BatchNormalization()(x)

    # Salida: (batch, window_size, filtros) — secuencia intacta para el BiLSTM
    return x

## Capa de Atención

In [8]:
#Capa de atención
class BahdanauAttention(layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=True)
        self.v = layers.Dense(1, use_bias=False)

    def call(self, h):
        score = self.v(tf.nn.tanh(self.W(h)))    # (batch, T, 1)
        weights = tf.nn.softmax(score, axis=1)      # (batch, T, 1)
        context = tf.reduce_sum(weights * h, axis=1) # (batch, dim)
        return context, weights

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"units": self.W.units})
        return cfg

## Construcción del modelo

In [9]:
def build_model(window_size=30, n_features=19,
                cnn_filtros=64, lstm_units=64,
                attn_units=64, dropout=0.3):

    inp = Input(shape=(window_size, n_features), name="entrada")

    # 1. CNN — patrones espaciales entre sensores
    x = build_cnn_block(inp, filtros=cnn_filtros)
    # (batch, 30, 64)

    # 2. BiLSTM — dependencias temporales en ambas direcciones
    x = layers.Bidirectional(
            layers.LSTM(lstm_units, return_sequences=True), name="bilstm_1")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Bidirectional(
            layers.LSTM(lstm_units, return_sequences=True), name="bilstm_2")(x)
    x = layers.Dropout(dropout)(x)
    # (batch, 30, 128)  ← 128 porque Bidirectional concatena fwd + bwd

    # 3. Atención — pondera timesteps por relevancia
    context, attn_weights = BahdanauAttention(units=attn_units, name="atencion")(x)
    # context: (batch, 128)

    # 4. Top Model
    x = layers.Dense(128, activation="relu")(context)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="relu")(x)
    rul = layers.Dense(1, name="rul")(x)

    model = Model(inputs=inp, outputs=rul, name="CNN_BiLSTM_Attn")
    return model


#model = build_model()
#model.summary()

In [10]:
# ── Pérdida asimétrica NASA ─────────────────────────────────────────────────
# Penaliza más predecir que queda MÁS vida de la que realmente queda
# (un fallo no anticipado es mucho más peligroso que una revisión anticipada)

def nasa_score_loss(y_true, y_pred):
    diff = y_pred - y_true
    score = tf.where(
        diff < 0,
        tf.exp(-diff / 13.0) - 1,   # predicción pesimista — penaliza suave
        tf.exp( diff / 10.0) - 1    # predicción optimista — penaliza fuerte
    )
    return tf.reduce_mean(score)

def rmse(y_true, y_pred):
    return tf.sqrt(tf.reduce_mean(tf.square(y_true - y_pred)))

In [11]:
# ── Preparar datos (del pipeline de preprocesamiento anterior) ──────────────

(X_train, y_train), (X_val, y_val), (X_test, y_test), scaler, feats = preparar_dataset()

# ── Compilar ────────────────────────────────────────────────────────────────

model = build_model(
    window_size = 30,
    n_features  = X_train.shape[2],   # se adapta automáticamente a los sensores útiles
    cnn_filtros = 64,
    lstm_units  = 64,
    attn_units  = 64,
    dropout     = 0.3
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor           = "val_loss",
        patience          = 20,
        restore_best_weights = True,
        verbose           = 1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor  = "val_loss",
        factor   = 0.5,
        patience = 8,
        min_lr   = 1e-6,
        verbose  = 1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath       = "best_model.keras",
        monitor        = "val_rmse",     # guardamos el mejor RMSE, no el mejor loss
        save_best_only = True,
        verbose        = 1
    ),
    tf.keras.callbacks.CSVLogger("historial_entrenamiento.csv")
]

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss      = nasa_score_loss,
    metrics   = [rmse, "mae"]
)

X_train: (15071, 30, 17)  y_train: (15071,)
X_val:   (2660, 30, 17)    y_val:   (2660,)
X_test:  (100, 30, 17)   y_test:  (100,)
Features usadas: 17 — ['op_1', 'op_2', 'op_3', 's2', 's3', 's4', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']



In [12]:
historial = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = 150,
    batch_size      = 256,
    callbacks       = callbacks,
    verbose         = 1
)

Epoch 1/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 14s 119ms/step - loss: 1102.1691 - mae: 46.7284 - rmse: 66.8957 - val_loss: 27.6845 - val_mae: 27.5358 - val_rmse: 50.7513
Epoch 2/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 97ms/step - loss: 13.6717 - mae: 23.7075 - rmse: 53.2077 - val_loss: 62.6445 - val_mae: 31.4402 - val_rmse: 43.4719
Epoch 3/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 104ms/step - loss: 10.5841 - mae: 20.9880 - rmse: 52.8873 - val_loss: 115.1376 - val_mae: 32.8533 - val_rmse: 43.9620
Epoch 4/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - loss: 9.5372 - mae: 19.7776 - rmse: 53.6705 - val_loss: 53.7359 - val_mae: 28.4292 - val_rmse: 46.3012
Epoch 5/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - loss: 8.9219 - mae: 19.0884 - rmse: 54.0960 - val_loss: 25.7478 - val_mae: 24.2828 - val_rmse: 48.4957
Epoch 6/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - loss: 7.4714 - mae: 17.8639 - rmse: 54.5964 - val_loss: 32.5555 - val_mae: 23.0177 - val_rmse: 50.7377
Epoch 7/150
59/59 ━━━━━━━━━━━━━━━━━━━━ 7s 113m